# 💻 Cheeni Capability 10: Natural Language to Code Executor
### Qwen2.5-Coder (`Qwen/Qwen2.5-Coder-1.5B-Instruct`)

**Core Capability**: Code Synthesis & Desktop Script Automation. Converts natural language instructions into executable Python, PowerShell, and Bash scripts, running them in an isolated subprocess.
**Foundation Architecture**: Qwen2.5-Coder-1.5B-Instruct (1.5B parameter specialized code LLM trained on 5.5 trillion tokens of code and technical text).
**Target Hugging Face Hub**: `wraithklu/cheeni-coder-qwen_10`
**Compute Environment**: Google Colab (Tesla T4 GPU or A100)

In [ ]:
# 1. Install Code LLM & Hugging Face Dependencies
!pip install -q transformers accelerate datasets huggingface_hub

In [ ]:
# 2. Authenticate with Hugging Face Hub
from huggingface_hub import login
login()

In [ ]:
# 3. Load Qwen2.5-Coder Architecture
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
print(f"⚡ Loading Qwen2.5-Coder model ({MODEL_ID})...")

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch_dtype,
    device_map="auto"
)
model.eval()
print(f"✅ Qwen2.5-Coder loaded on {device} ({torch_dtype})!")

In [ ]:
# 4. Define Helper Function: Generate Clean Code from Spoken Command
def generate_code(user_prompt: str) -> str:
    messages = [
        {"role": "system", "content": "You are Cheeni's expert code executor. Output ONLY executable Python code solving the user request without markdown backticks or commentary."},
        {"role": "user", "content": user_prompt}
    ]
    
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.2,
            top_p=0.9,
            do_sample=True
        )
    
    generated_ids = [output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response.strip()

print("✅ Code generation function ready!")

In [ ]:
# 5. Test Live Code Generation & In-Memory Execution
import subprocess
import sys

test_prompt = "Calculate the sum of squares of all even numbers from 1 to 20 and print the result."
print("=" * 65)
print(f"📝 Prompt: '{test_prompt}'")
print("=" * 65)

generated_script = generate_code(test_prompt)
print(f"💻 Generated Python Code:\n{generated_script}")
print("=" * 65)

# Clean any residual markdown formatting if model returned it
clean_script = generated_script.replace("```python", "").replace("```", "").strip()

# Execute safely via subprocess
result = subprocess.run([sys.executable, "-c", clean_script], capture_output=True, text=True, timeout=5)
print(f"⚡ Execution Output (stdout):\n{result.stdout.strip()}")
if result.stderr:
    print(f"⚠️ Stderr:\n{result.stderr.strip()}")
print("=" * 65)

In [ ]:
# 6. Push Verified Qwen-Coder Model & Tokenizer to Hugging Face Hub
HF_REPO_ID = "wraithklu/cheeni-coder-qwen_10"

print(f"Pushing trained Code Model to https://huggingface.co/{HF_REPO_ID}...")
model.push_to_hub(HF_REPO_ID)
tokenizer.push_to_hub(HF_REPO_ID)
print(f"🎉 SUCCESS! Model is live on Hugging Face: https://huggingface.co/{HF_REPO_ID}")